In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='mps')

In [2]:
ds = load_dataset("imdb")
ds

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
import re
from collections import Counter

def simple_tokenize(text: str):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    # keep words and basic punctuation as separate tokens
    tokens = re.findall(r"[a-z0-9']+|[.,!?;]", text)
    return tokens

# Build vocab from training set (optionally limit size)
MAX_VOCAB = 20000
MIN_FREQ = 2
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

counter = Counter()
for ex in tqdm(ds["train"], desc="Building vocab"):
    counter.update(simple_tokenize(ex["text"]))

# keep tokens with freq >= MIN_FREQ
tokens_and_freqs = [(t, f) for t, f in counter.items() if f >= MIN_FREQ]
tokens_and_freqs.sort(key=lambda x: x[1], reverse=True)

vocab_tokens = [PAD_TOKEN, UNK_TOKEN] + [t for t, _ in tokens_and_freqs[:MAX_VOCAB-2]]
stoi = {t: i for i, t in enumerate(vocab_tokens)}
itos = vocab_tokens

pad_id = stoi[PAD_TOKEN]
unk_id = stoi[UNK_TOKEN]

len(itos), itos[:20]

Building vocab:   0%|          | 0/25000 [00:00<?, ?it/s]

(20000,
 ['<pad>',
  '<unk>',
  'the',
  '.',
  ',',
  'and',
  'a',
  'of',
  'to',
  'is',
  'in',
  'it',
  'i',
  'this',
  'that',
  'was',
  'as',
  'for',
  'with',
  'movie'])

In [4]:
MAX_LEN = 256  # truncate long reviews

def encode(text: str):
    toks = simple_tokenize(text)[:MAX_LEN]
    ids = [stoi.get(t, unk_id) for t in toks]
    return ids

def collate_batch(batch):
    # batch is a list of dicts with keys: "text", "label"
    sequences = [encode(x["text"]) for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)

    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    max_len = lengths.max().item() if len(lengths) > 0 else 0

    padded = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(sequences):
        if len(seq) > 0:
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)

    return padded, lengths, labels

In [5]:
BATCH_SIZE = 64

train_loader = DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_loader  = DataLoader(ds["test"],  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

In [6]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,   # <- unidirectional
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        # x: (B, T)
        emb = self.embedding(x)  # (B, T, E)

        # Pack padded sequence
        lengths_cpu = lengths.cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed)

        # h_n: (num_layers * num_directions, B, hidden_dim)
        last_hidden = h_n[-1]  # (B, hidden_dim) since unidirectional
        out = self.dropout(last_hidden)
        logits = self.fc(out).squeeze(1)  # (B,)
        return logits

model = SentimentLSTM(vocab_size=len(itos), emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(device)
model

SentimentLSTM(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [7]:
def accuracy_from_logits(logits, labels):
    # logits: real numbers, labels: 0/1
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [8]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Train", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Eval", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

In [9]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Train", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Eval", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

In [ ]:
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} | test loss {test_loss:.4f} acc {test_acc:.4f}")

Train:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
@torch.no_grad()
def predict_sentiment(text: str):
    model.eval()
    ids = encode(text)
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)], dtype=torch.long).to(device)
    logits = model(x, lengths)
    prob = torch.sigmoid(logits).item()
    return prob

texts = [
    "This movie was fantastic, I loved it!",
    "Boring and too long. I regret watching it."
]
for t in texts:
    p = predict_sentiment(t)
    print(f"{p:.3f} -> {'POS' if p>=0.5 else 'NEG'} | {t}")